In [ ]:
### Analysis of the relationship of the different protein identification rates to the proteome size, genus and species within genus ###

import os
import pandas as pd
from scipy import stats
from pathlib import Path
import numpy as np
import statsmodels.formula.api as smf

In [35]:
root = Path.cwd()

In [ ]:
# Both tables from the github repository
excel_100 = pd.read_excel(root / "DIA_and_DDA_QC_metrics.xlsx", sheet_name=["DIA100", "DIA200", "DIA500", "DDA100"])
excel_meta = pd.read_excel(root / "full_metadata_table.xlsx") 

In [37]:
genus_list = [x for x in excel_meta["Genus"].to_list() if str(x) != "nan"]
species_list = [x for x in excel_meta["Species / Subspecies"].to_list() if str(x) != "nan"]

In [38]:
### Spearman rank-order correlation coefficient ###
### Calculated to assess if there is a rank-order correlation between number of proteins identified and the reference proteome ###

for key in excel_100.keys():
    df = excel_100[key]

    if "DIA" in key:
        rho, p = stats.spearmanr(df["Total.Protein.Number.Ref"], df["Proteins.Identified"])
    else:
        rho, p = stats.spearmanr(df["Total.Protein.Number.Ref"], df["Identified_proteins"])

    print(f"ID'd proteins vs proteome size for the dataset {key}: Spearman rho={rho:.2f}, p={p:.2g}")

ID'd proteins vs proteome size for the dataset DIA100: Spearman rho=0.78, p=1.6e-42
ID'd proteins vs proteome size for the dataset DIA200: Spearman rho=0.70, p=7.5e-31
ID'd proteins vs proteome size for the dataset DIA500: Spearman rho=0.57, p=2.9e-18
ID'd proteins vs proteome size for the dataset DDA100: Spearman rho=0.77, p=7.6e-40


In [39]:
### Here we fit log transformed values using linear regression and can see there is a relationship between ###
### identified proteins and size of the reference proteome but not a proportional one ###

for key in excel_100.keys():
    df = excel_100[key]
    ids = df["Proteins.Identified"] if "DIA" in key else df["Identified_proteins"]
    res = stats.linregress(np.log10(df["Total.Protein.Number.Ref"]), np.log10(ids))
    lo, hi = res.slope - 1.96 * res.stderr, res.slope + 1.96 * res.stderr
    print(f"{key}: log-log slope={res.slope:.2f} (95% CI {lo:.2f}-{hi:.2f}), R²={res.rvalue**2:.2f}")

DIA100: log-log slope=0.55 (95% CI 0.49-0.61), R²=0.62
DIA200: log-log slope=0.43 (95% CI 0.36-0.49), R²=0.46
DIA500: log-log slope=0.31 (95% CI 0.24-0.38), R²=0.29
DDA100: log-log slope=0.49 (95% CI 0.43-0.54), R²=0.62


In [43]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.simplefilter("ignore", ConvergenceWarning) # Has been tested and its not the case, its only supressed as internal paths have been printed

for key in excel_100.keys():
    df = excel_100[key].copy()
    df["Genus"] = genus_list
    df["Species / Subspecies"] = species_list
    id_col = "Proteins.Identified" if "DIA" in key else "Identified_proteins"
    df["log_size"] = np.log10(df["Total.Protein.Number.Ref"])
    df["log_ids"] = np.log10(df[id_col])

    df["Species"] = df["Species / Subspecies"].str.strip().str.split().str[0]
    # Nested random effects: genus, and species within genus
    m = smf.mixedlm(
        "log_ids ~ log_size", df,
        groups="Genus",
        re_formula="1",
        vc_formula={"Species": "0 + C(Species)"},
    ).fit(reml=True)

    var_genus = float(m.cov_re.iloc[0, 0])
    var_species = float(m.vcomp[0])
    var_resid = float(m.scale)
    total = var_genus + var_species + var_resid

    print(f"\n{key}: slope={m.fe_params['log_size']:.2f}")
    print(f"  genus:           {var_genus / total:.2f}")
    print(f"  species (in genus): {var_species / total:.2f}")
    print(f"  residual (samples): {var_resid / total:.2f}")


DIA100: slope=0.54
  genus:           0.18
  species (in genus): 0.69
  residual (samples): 0.13

DIA200: slope=0.42
  genus:           0.19
  species (in genus): 0.59
  residual (samples): 0.23

DIA500: slope=0.30
  genus:           0.22
  species (in genus): 0.45
  residual (samples): 0.33

DDA100: slope=0.47
  genus:           0.33
  species (in genus): 0.39
  residual (samples): 0.28


In [42]:
import warnings
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.sm_exceptions import ConvergenceWarning


def fit_mixed(formula, data, **model_kwargs): ### Fitting  a REML mixed model; ConvergenceWarning silenced (we already verified that estimates were stable across optimizers) ##

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        return smf.mixedlm(formula, data, **model_kwargs).fit(reml=True, method="lbfgs")


results = []

for key in excel_100.keys():
    df = excel_100[key].copy()
    df["Genus"] = genus_list
    df["Species / Subspecies"] = species_list
    id_col = "Proteins.Identified" if "DIA" in key else "Identified_proteins"
    df["log_size"] = np.log10(df["Total.Protein.Number.Ref"])
    df["log_ids"] = np.log10(df[id_col])
    df["Species"] = df["Species / Subspecies"].str.strip().str.split().str[0]
    df["Genus_Species"] = df["Genus"] + "_" + df["Species"]

    # Full model: genus + species within genus
    m_full = fit_mixed("log_ids ~ log_size", df,
                       groups="Genus", re_formula="1",
                       vc_formula={"Species": "0 + C(Species)"})

    # Reduced model: species only, no genus level
    m_no_genus = fit_mixed("log_ids ~ log_size", df, groups="Genus_Species")

    # Variance components from the full model
    var_genus = float(m_full.cov_re.iloc[0, 0])
    var_species = float(m_full.vcomp[0])
    var_strain = float(m_full.scale)
    total = var_genus + var_species + var_strain

    # Likelihood ratio test for the genus level
    lr = 2 * (m_full.llf - m_no_genus.llf)
    p_genus = 0.5 * stats.chi2.sf(lr, df=1)

    results.append({
        "Dataset": key,
        "Slope_mixed": m_full.fe_params["log_size"],
        "Genus": var_genus / total,
        "Species": var_species / total,
        "Strain": var_strain / total,
        "LR": lr,
        "p_genus": p_genus,
    })

results = pd.DataFrame(results)
print(results.round({"Slope_mixed": 2, "Genus": 2, "Species": 2, "Strain": 2, "LR": 2, "p_genus": 4}).to_string(index=False))

Dataset  Slope_mixed  Genus  Species  Strain    LR  p_genus
 DIA100         0.54   0.18     0.69    0.13  4.53   0.0167
 DIA200         0.42   0.19     0.59    0.23  4.50   0.0169
 DIA500         0.30   0.22     0.45    0.33  5.92   0.0075
 DDA100         0.47   0.33     0.39    0.28 12.26   0.0002
